# Synthetic benchmark: 40- and 50-member knee extensions

Runs only the new 40- and 50-member conditions. Both forward and backward
searches use their existing optimized search procedures, then select the knee,
retune the knee subset, evaluate it on the fixed blind set, and fit a matched
sklearn SVM on its top unified-ranked features, capped at the knee feature count.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Make the notebook work from either the repository root or validation.
repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "mistic" / "svmSet.py").exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mistic import combined_rank, cvSet, kernelWrapper, paramSet, score_svc, svmSet

sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
N_SAMPLES = 500
N_FEATURES = 100
N_INFORMATIVE = 10
N_REDUNDANT = 10
SIGNAL_FEATURES = set(range(N_INFORMATIVE + N_REDUNDANT))

X_values, y_values = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    n_redundant=N_REDUNDANT,
    n_repeated=0,
    n_classes=2,
    weights=[0.55, 0.45],
    class_sep=1.0,
    flip_y=0.03,
    shuffle=False,
    random_state=2026,
)
feature_names = [
    *(f"informative_{i:02d}" for i in range(N_INFORMATIVE)),
    *(f"redundant_{i:02d}" for i in range(N_REDUNDANT)),
    *(f"noise_{i:02d}" for i in range(N_FEATURES - N_INFORMATIVE - N_REDUNDANT)),
]
X = pd.DataFrame(X_values, columns=feature_names)
y = pd.Series(y_values, name="class")

BASE_MODEL_COUNTS = [1, 3, 5, 10, 20]
MODEL_COUNTS = [*BASE_MODEL_COUNTS, 30]
BLIND_SET_SEED = 42
INNER_SEEDS = list(range(3))
TEST_SIZE = 0.25
INNER_VALIDATION_SIZE = 0.20
RANK_WEIGHT = 0.75
SELECTION_STRATEGIES = ["backward"]

# A compact grid keeps the full selection-by-size experiment tractable.
C_VALUES = [0.25, 1.0, 4.0]
GAMMA_VALUES = [2.0 ** exponent for exponent in (-9, -7, -5)]

print(f"Samples: {len(X)}, features: {X.shape[1]}")
print(y.value_counts().sort_index())

In [ ]:
def metric_row(y_true, predictions, decision_values):
    # Metrics computed only from the untouched blind-set observations.
    return {
        "roc_auc": roc_auc_score(y_true, decision_values),
        "f1": f1_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "accuracy": accuracy_score(y_true, predictions),
    }


def fit_sklearn_pipeline(X_train, y_train, seed):
    pipeline = Pipeline([
        ("scale", StandardScaler()),
        ("svc", SVC(kernel="rbf", class_weight="balanced")),
    ])
    search = GridSearchCV(
        pipeline,
        param_grid={"svc__C": C_VALUES, "svc__gamma": GAMMA_VALUES},
        scoring="roc_auc",
        cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=seed),
        n_jobs=-1,
        refit=True,
    )
    return search.fit(X_train, y_train)


def fit_svm_set(X_train, y_train, num_models, seed, selection_strategy):
    # Fit the scaler on this outer-training split only.
    scaler = StandardScaler().fit(X_train)
    X_scaled = scaler.transform(X_train)

    splits = cvSet(X_scaled, np.asarray(y_train))
    splits.classification(
        num_sets=num_models,
        validation_size=INNER_VALIDATION_SIZE,
        random_seed=seed,
    )

    ensemble = svmSet(
        SVC(kernel="precomputed", class_weight="balanced"),
        splits,
        score_method=score_svc(weight=1.0).score,  # tune for ROC AUC
        kernel=kernelWrapper(type="rbf"),
        separate_feature_sets=True,
        separate_parameters=True,
    )
    parameter_grid = [
        paramSet(model={"C": cost}, kernel={"gamma": gamma})
        for cost in C_VALUES
        for gamma in GAMMA_VALUES
    ]
    selection_options = dict(
        parameter_grid=parameter_grid,
        feature_ranker=combined_rank(weight=RANK_WEIGHT).compute,
        set_for_rank="sample",
    )
    if selection_strategy != "backward":
        raise ValueError(f"unknown selection strategy: {selection_strategy}")
    ensemble.greedy_backward_selection(
        reduction_factor=0.1,
        tune_models_each_step=False,
        **selection_options,
    )
    return scaler, ensemble

In [ ]:
import copy

FORWARD_MODEL_COUNTS = [1, 3, 5, 10, 20, 30]
FORWARD_MAX_FEATURES = 20


def fit_forward_svm_set(X_train, y_train, num_models, seed):
    scaler = StandardScaler().fit(X_train)
    X_scaled = scaler.transform(X_train)
    splits = cvSet(X_scaled, np.asarray(y_train), num_feature_medoids=20)
    splits.classification(
        num_sets=num_models,
        validation_size=INNER_VALIDATION_SIZE,
        random_seed=seed,
    )
    ensemble = svmSet(
        SVC(kernel="precomputed", class_weight="balanced"),
        splits,
        score_method=score_svc(weight=1.0).score,
        kernel=kernelWrapper(type="rbf"),
        separate_feature_sets=True,
        separate_parameters=True,
    )
    parameter_grid = [
        paramSet(model={"C": cost}, kernel={"gamma": gamma})
        for cost in C_VALUES
        for gamma in GAMMA_VALUES
    ]
    ensemble.greedy_forward_selection(
        parameter_grid=parameter_grid,
        reduction_factor=0,
        feature_ranker=combined_rank(weight=RANK_WEIGHT).compute,
        set_for_rank="sample",
        tune_models_each_step=False,
        max_features=FORWARD_MAX_FEATURES,
    )
    return scaler, ensemble, parameter_grid


def top20_recovery(ensemble):
    unified = np.asarray(ensemble.unified_features, dtype=int)
    unified_set = set(unified)
    ranked = np.asarray([
        feature for feature in ensemble.unified_sorted_features
        if feature in unified_set
    ], dtype=int)
    top = ranked[:20]
    selected_signal = len(set(top).intersection(SIGNAL_FEATURES))
    return unified, selected_signal / len(SIGNAL_FEATURES), 1 - selected_signal / len(top)


def append_condition(rows, ensemble, scaler, X_train, X_blind, y_train,
                     y_blind, seed, num_models, strategy, knee_feature_count=None):
    X_blind_scaled = scaler.transform(X_blind)
    decisions = ensemble.decision_function(X_blind_scaled)
    predictions = ensemble.predict(X_blind_scaled)
    member_predictions = np.column_stack([
        ensemble.predict(X_blind_scaled, model_index=index)
        for index in range(num_models)
    ])
    pairs = [
        np.mean(member_predictions[:, left] != member_predictions[:, right])
        for left in range(num_models)
        for right in range(left + 1, num_models)
    ]
    unified, signal_recall, noise_fraction = top20_recovery(ensemble)
    rows.append({
        "seed": seed,
        "method": "MiSTIC SVM set",
        "selection_strategy": strategy,
        "num_models": num_models,
        "num_unified_features": len(unified),
        "signal_recall": signal_recall,
        "noise_fraction": noise_fraction,
        "member_disagreement": float(np.mean(pairs)) if pairs else 0.0,
        **metric_row(y_blind, predictions, decisions),
    })

    if knee_feature_count is None:
        reference_features = unified
    else:
        unified_set = set(unified)
        ranked_unified = np.asarray([
            feature for feature in ensemble.unified_sorted_features
            if feature in unified_set
        ], dtype=int)
        reference_features = ranked_unified[:min(knee_feature_count, len(ranked_unified))]
    reference_signal = len(set(reference_features).intersection(SIGNAL_FEATURES))
    reference = fit_sklearn_pipeline(
        X_train.iloc[:, reference_features], y_train, seed
    )
    rows.append({
        "seed": seed,
        "method": "sklearn on knee-ranked unified features",
        "selection_strategy": strategy,
        "num_models": num_models,
        "num_unified_features": len(reference_features),
        "signal_recall": reference_signal / len(SIGNAL_FEATURES),
        "noise_fraction": 1 - reference_signal / len(reference_features),
        "member_disagreement": 0.0,
        **metric_row(
            y_blind,
            reference.predict(X_blind.iloc[:, reference_features]),
            reference.decision_function(X_blind.iloc[:, reference_features]),
        ),
    })

In [ ]:
KNEE_EXTENSION_COUNTS = [40, 50]
X_train, X_blind, y_train, y_blind = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=BLIND_SET_SEED,
)

## Forward-knee conditions

In [ ]:
forward_knee_rows = []
for seed in INNER_SEEDS:
    for num_models in KNEE_EXTENSION_COUNTS:
        scaler, best_ensemble, parameter_grid = fit_forward_svm_set(
            X_train, y_train, num_models, seed
        )
        knee_ensemble = copy.deepcopy(best_ensemble)
        knee_count = knee_ensemble.find_knee(metric="score")
        knee_ensemble.set_num_features(knee_count, parameter_grid)
        append_condition(
            forward_knee_rows, knee_ensemble, scaler, X_train, X_blind,
            y_train, y_blind, seed, num_models, "forward_knee", knee_feature_count=knee_count,
        )
        forward_knee_rows[-2]["knee_feature_count"] = knee_count
        forward_knee_rows[-1]["knee_feature_count"] = knee_count
        print(f"completed forward seed {seed}, {num_models} models; knee={knee_count}")

forward_knee_extension = pd.DataFrame(forward_knee_rows)
forward_knee_extension

## Backward-knee conditions

In [ ]:
backward_knee_rows = []
for seed in INNER_SEEDS:
    for num_models in KNEE_EXTENSION_COUNTS:
        scaler, best_ensemble = fit_svm_set(
            X_train, y_train, num_models, seed, "backward"
        )
        parameter_grid = [
            paramSet(model={"C": cost}, kernel={"gamma": gamma})
            for cost in C_VALUES
            for gamma in GAMMA_VALUES
        ]
        knee_ensemble = copy.deepcopy(best_ensemble)
        knee_count = knee_ensemble.find_knee(metric="score")
        knee_ensemble.set_num_features(knee_count, parameter_grid)
        append_condition(
            backward_knee_rows, knee_ensemble, scaler, X_train, X_blind,
            y_train, y_blind, seed, num_models, "backward_knee", knee_feature_count=knee_count,
        )
        backward_knee_rows[-2]["knee_feature_count"] = knee_count
        backward_knee_rows[-1]["knee_feature_count"] = knee_count
        print(f"completed backward seed {seed}, {num_models} models; knee={knee_count}")

backward_knee_extension = pd.DataFrame(backward_knee_rows)
backward_knee_extension

In [ ]:
knee_extension_results = pd.concat(
    [forward_knee_extension, backward_knee_extension], ignore_index=True
)
output_path = repo_root / "validation/Synthetic100_svmSet_knee_40_50_results.csv"
knee_extension_results.to_csv(output_path, index=False)
knee_extension_results.groupby(
    ["method", "selection_strategy", "num_models"]
).agg(
    mean_auc=("roc_auc", "mean"),
    mean_f1=("f1", "mean"),
    mean_accuracy=("accuracy", "mean"),
    mean_feature_count=("num_unified_features", "mean"),
    mean_signal_recall=("signal_recall", "mean"),
).round(4)